# cellPMVI: Multi-modal VAE for CITE-seq Data

This tutorial demonstrates how to use **cellPMVI** — a multi-modal variational autoencoder for joint analysis of single-cell RNA and surface protein (CITE-seq) data.

We use the [10x PBMC CITE-seq dataset](https://www.10xgenomics.com/datasets) bundled with scvi-tools, containing ~10,800 PBMCs profiled with RNA-seq and 14 TotalSeq-B surface protein antibodies across two batches.

**What you'll learn:**
1. Data loading and preprocessing
2. Setting up and training a cellPMVI model
3. Latent space visualization
4. Reconstruction quality assessment
5. Prior and posterior predictive sampling
6. Comparing posterior fusion strategies (PoE, MoE, None)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import scvi
import torch

from cellpmvi import CellPMVI
from cellpmvi.utils import compute_reconstruction_metrics, compute_latent_metrics

sc.set_figure_params(dpi=100, frameon=False)
scvi.settings.seed = 0
torch.set_float32_matmul_precision("high")

%matplotlib inline

## 1. Data Loading

We load the 10x PBMC CITE-seq dataset provided by scvi-tools. This contains two batches of PBMCs (PBMC10k and PBMC5k), each profiled with both RNA-seq and 14 surface protein antibodies.

In [ ]:
adata = scvi.data.pbmcs_10x_cite_seq(save_path="data/")
adata.obs_names_make_unique()

print(f"Cells: {adata.n_obs:,}")
print(f"Genes: {adata.n_vars:,}")
print(f"Proteins: {adata.obsm['protein_expression'].shape[1]}")
print(f"\nBatch distribution:")
print(adata.obs["batch"].value_counts())
print(f"\nProtein panel:")
print(list(adata.obsm["protein_expression"].columns))

## 2. Preprocessing

We apply standard preprocessing: filter low-count genes, select highly variable genes (HVGs), and store raw counts for model input. cellPMVI models raw counts directly using a negative binomial likelihood.

In [ ]:
# Filter low-count genes
sc.pp.filter_genes(adata, min_counts=3)

# Store raw counts before normalization
adata.layers["counts"] = adata.X.copy()

# Normalize and log-transform for HVG selection
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

# Select highly variable genes using raw counts (seurat_v3)
sc.pp.highly_variable_genes(
    adata, n_top_genes=4000, flavor="seurat_v3", layer="counts"
)
print(f"HVGs selected: {adata.var['highly_variable'].sum()}")

# Subset to HVGs and restore raw counts in .X
adata = adata[:, adata.var.highly_variable].copy()
adata.X = adata.layers["counts"].copy()

print(f"Final: {adata.n_obs:,} cells x {adata.n_vars:,} genes")

### Cell type annotation via clustering

The dataset does not include cell type labels, so we perform Leiden clustering on the normalized data and use protein surface markers to annotate major cell populations.

In [ ]:
# Run PCA, neighbors, UMAP, and Leiden on log-normalized data
adata_pp = adata.copy()
sc.pp.normalize_total(adata_pp)
sc.pp.log1p(adata_pp)
sc.pp.scale(adata_pp, max_value=10)
sc.tl.pca(adata_pp, n_comps=30)
sc.pp.neighbors(adata_pp, n_pcs=30)
sc.tl.umap(adata_pp)
sc.tl.leiden(adata_pp, resolution=1.0, key_added="leiden")

# Transfer to main adata
adata.obs["leiden"] = adata_pp.obs["leiden"]
adata.obsm["X_umap_raw"] = adata_pp.obsm["X_umap"]

In [ ]:
# Visualize clusters alongside protein markers
# Use '_protein' suffix to avoid clashing with gene names in .var_names
protein_df = adata.obsm["protein_expression"]
markers = {
    "CD3_protein": "CD3_TotalSeqB",    # T cells
    "CD4_protein": "CD4_TotalSeqB",    # CD4 T cells
    "CD8a_protein": "CD8a_TotalSeqB",  # CD8 T cells
    "CD14_protein": "CD14_TotalSeqB",  # Monocytes
    "CD19_protein": "CD19_TotalSeqB",  # B cells
    "CD56_protein": "CD56_TotalSeqB",  # NK cells
}
for short, full in markers.items():
    adata.obs[short] = protein_df[full].values.astype(float)

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
sc.pl.embedding(adata, basis="X_umap_raw", color="leiden", ax=axes[0, 0], show=False, title="Leiden")
sc.pl.embedding(adata, basis="X_umap_raw", color="batch", ax=axes[0, 1], show=False, title="Batch")
for ax, marker in zip(axes.flat[2:], markers.keys()):
    label = marker.replace("_protein", "")
    sc.pl.embedding(adata, basis="X_umap_raw", color=marker, ax=ax, show=False,
                    title=label, color_map="viridis")
plt.tight_layout()
plt.show()

In [ ]:
# Compute per-cluster mean protein expression for annotation
cluster_protein = pd.DataFrame(index=sorted(adata.obs["leiden"].unique(), key=int))
for short, full in markers.items():
    cluster_protein[short] = adata.obs.groupby("leiden")[short].mean()

# Display with clean column names
display_df = cluster_protein.copy()
display_df.columns = [c.replace("_protein", "") for c in display_df.columns]
print("Mean protein expression per cluster:")
print(display_df.round(1).to_string())

In [ ]:
# Annotate cell types based on dominant protein markers
# Adjust this mapping based on the cluster-protein table above
def annotate_clusters(row):
    """Rule-based annotation using protein surface markers."""
    if row["CD19_protein"] > 30:
        return "B cells"
    if row["CD14_protein"] > 30:
        return "CD14 Mono"
    if row["CD56_protein"] > 30 and row["CD3_protein"] < 20:
        return "NK"
    if row["CD3_protein"] > 20 and row["CD8a_protein"] > 20:
        return "CD8 T"
    if row["CD3_protein"] > 20 and row["CD4_protein"] > 20:
        return "CD4 T"
    if row["CD3_protein"] > 20:
        return "T cells"
    return "Other"

cluster_to_celltype = cluster_protein.apply(annotate_clusters, axis=1).to_dict()
adata.obs["cell_type"] = adata.obs["leiden"].map(cluster_to_celltype).astype("category")

print("Cell type counts:")
print(adata.obs["cell_type"].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sc.pl.embedding(adata, basis="X_umap_raw", color="cell_type", ax=axes[0], show=False, title="Cell Types")
sc.pl.embedding(adata, basis="X_umap_raw", color="batch", ax=axes[1], show=False, title="Batch")
plt.tight_layout()
plt.show()

## 3. Model Setup and Training

cellPMVI uses separate encoder/decoder pairs for RNA and protein modalities, with optional posterior fusion via **Product-of-Experts (PoE)** or **Mixture-of-Experts (MoE)**.

Key parameters:
- `gene_likelihood="nb"` — Negative binomial for RNA counts
- `fusion_method="poe"` — Combine modality-specific posteriors into a sharper joint posterior
- `n_latent=20` — Dimensionality of the shared latent space

In [ ]:
CellPMVI.setup_anndata(
    adata,
    layer="counts",
    protein_expression_obsm_key="protein_expression",
    batch_key="batch",
)

In [ ]:
model = CellPMVI(
    adata,
    n_hidden=128,
    n_latent=20,
    n_layers=2,
    dropout_rate=0.1,
    gene_likelihood="nb",
    latent_distribution="normal",
    fusion_method="poe",
)
print(model._model_summary_string)

In [ ]:
model.train(max_epochs=200, batch_size=256)

In [ ]:
# Plot training loss
train_loss = model.history["train_loss"]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(train_loss.index, train_loss["train_loss"])
ax.set_xlabel("Epoch")
ax.set_ylabel("ELBO Loss")
ax.set_title("cellPMVI Training Convergence")
plt.tight_layout()
plt.show()

## 4. Latent Space Visualization

We extract the learned latent representation and visualize it with UMAP, colored by cell type and batch. A good multi-modal integration should:
- Separate cell types (high bio-conservation)
- Mix batches within cell types (good batch correction)

In [ ]:
latent = model.get_latent_representation()
adata.obsm["X_cellpmvi"] = latent
print(f"Latent shape: {latent.shape}")

# Compute UMAP on cellPMVI latent space
sc.pp.neighbors(adata, use_rep="X_cellpmvi", key_added="cellpmvi")
sc.tl.umap(adata, neighbors_key="cellpmvi", key_added="X_umap_cellpmvi")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sc.pl.embedding(adata, basis="X_umap_cellpmvi", color="cell_type",
                ax=axes[0], show=False, title="cellPMVI — Cell Type")
sc.pl.embedding(adata, basis="X_umap_cellpmvi", color="batch",
                ax=axes[1], show=False, title="cellPMVI — Batch")
plt.tight_layout()
plt.show()

In [ ]:
# Quantify latent space quality
metrics = compute_latent_metrics(
    latent,
    labels=adata.obs["cell_type"].values,
    batch=adata.obs["batch"].values,
)
print("Latent space metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

## 5. Reconstruction Quality

We assess how well the model reconstructs the observed data by sampling from the posterior predictive distribution and comparing against the original counts.

In [ ]:
rna_samples, protein_samples = model.posterior_predictive_sample(n_samples=1)
print(f"RNA reconstruction: {rna_samples.shape}")
print(f"Protein reconstruction: {protein_samples.shape}")

In [ ]:
# RNA reconstruction metrics
rna_obs = np.array(adata.layers["counts"])
rna_metrics = compute_reconstruction_metrics(rna_obs, rna_samples)
print("RNA reconstruction:")
for k, v in rna_metrics.items():
    print(f"  {k}: {v:.4f}")

# Protein reconstruction metrics
protein_obs = adata.obsm["protein_expression"].values.astype(float)
protein_metrics = compute_reconstruction_metrics(protein_obs, protein_samples)
print("\nProtein reconstruction:")
for k, v in protein_metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# RNA: observed vs predicted gene means
obs_mean = rna_obs.mean(axis=0).ravel()
pred_mean = rna_samples.mean(axis=0).ravel()
axes[0].scatter(np.log1p(obs_mean), np.log1p(pred_mean), s=1, alpha=0.3)
lim = max(np.log1p(obs_mean).max(), np.log1p(pred_mean).max())
axes[0].plot([0, lim], [0, lim], "r--", alpha=0.5)
axes[0].set_xlabel("Observed log(mean + 1)")
axes[0].set_ylabel("Predicted log(mean + 1)")
axes[0].set_title(f"RNA Gene Means (\u03C1={rna_metrics['spearman_mean']:.3f})")

# Protein: observed vs predicted per-protein means
obs_p = protein_obs.mean(axis=0)
pred_p = protein_samples.mean(axis=0)
axes[1].scatter(obs_p, pred_p, s=50, zorder=5)
for i, name in enumerate(adata.obsm["protein_expression"].columns):
    short = name.replace("_TotalSeqB", "")
    axes[1].annotate(short, (obs_p[i], pred_p[i]), fontsize=8,
                     xytext=(5, 5), textcoords="offset points")
lim = max(obs_p.max(), pred_p.max()) * 1.1
axes[1].plot([0, lim], [0, lim], "r--", alpha=0.5)
axes[1].set_xlabel("Observed mean")
axes[1].set_ylabel("Predicted mean")
axes[1].set_title(f"Protein Means (\u03C1={protein_metrics['spearman_mean']:.3f})")

plt.tight_layout()
plt.show()

## 6. Prior Predictive Sampling

We can generate entirely new synthetic cells by sampling from the prior distribution. This is useful for assessing whether the learned generative model captures realistic expression patterns.

In [ ]:
prior_rna, prior_protein = model.prior_predictive_sample(n_samples=500)
print(f"Prior RNA samples: {prior_rna.shape}")
print(f"Prior Protein samples: {prior_protein.shape}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# RNA library sizes
obs_lib = np.array(adata.layers["counts"].sum(axis=1)).ravel()
prior_lib = prior_rna.sum(axis=1)
axes[0].hist(np.log10(obs_lib + 1), bins=50, alpha=0.6, label="Observed", density=True)
axes[0].hist(np.log10(prior_lib + 1), bins=50, alpha=0.6, label="Prior", density=True)
axes[0].set_xlabel("log10(library size + 1)")
axes[0].set_ylabel("Density")
axes[0].set_title("RNA Library Size")
axes[0].legend()

# Protein library sizes
obs_plib = protein_obs.sum(axis=1)
prior_plib = prior_protein.sum(axis=1)
axes[1].hist(np.log10(obs_plib + 1), bins=50, alpha=0.6, label="Observed", density=True)
axes[1].hist(np.log10(prior_plib + 1), bins=50, alpha=0.6, label="Prior", density=True)
axes[1].set_xlabel("log10(library size + 1)")
axes[1].set_ylabel("Density")
axes[1].set_title("Protein Library Size")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Comparing Fusion Strategies

cellPMVI supports three posterior fusion strategies:

- **Product-of-Experts (PoE)**: Combines modality posteriors multiplicatively → sharper joint posterior
- **Mixture-of-Experts (MoE)**: Averages posteriors → broader, more robust to missing modalities
- **None**: Independent latent spaces per modality with cross-modal reconstruction

We train all three and compare their latent representations.

In [ ]:
fusion_results = {}

for fusion in ["poe", "moe", "none"]:
    print(f"\n{'='*50}")
    print(f"Training with fusion_method='{fusion}'")
    print(f"{'='*50}")
    
    # Re-register adata for each new model instance
    CellPMVI.setup_anndata(
        adata, layer="counts",
        protein_expression_obsm_key="protein_expression",
        batch_key="batch",
    )
    m = CellPMVI(
        adata, n_hidden=128, n_latent=20, n_layers=2,
        gene_likelihood="nb", fusion_method=fusion,
    )
    m.train(max_epochs=200, batch_size=256)
    
    lat = m.get_latent_representation()
    met = compute_latent_metrics(
        lat,
        labels=adata.obs["cell_type"].values,
        batch=adata.obs["batch"].values,
    )
    
    # Posterior reconstruction
    rna_s, pro_s = m.posterior_predictive_sample(n_samples=1)
    rna_met = compute_reconstruction_metrics(rna_obs, rna_s)
    pro_met = compute_reconstruction_metrics(protein_obs, pro_s)
    
    fusion_results[fusion] = {
        "latent": lat,
        "latent_metrics": met,
        "rna_recon": rna_met,
        "protein_recon": pro_met,
        "history": m.history["train_loss"],
    }
    
    print(f"  Latent: {met}")
    print(f"  RNA recon: spearman_mean={rna_met['spearman_mean']:.4f}")
    print(f"  Protein recon: spearman_mean={pro_met['spearman_mean']:.4f}")

In [ ]:
# Compare latent spaces visually
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, (fusion, res) in zip(axes, fusion_results.items()):
    adata.obsm["X_tmp"] = res["latent"]
    sc.pp.neighbors(adata, use_rep="X_tmp", key_added="tmp")
    sc.tl.umap(adata, neighbors_key="tmp", key_added="X_umap_tmp")
    
    sil = res["latent_metrics"].get("silhouette_labels", 0)
    rna_rho = res["rna_recon"]["spearman_mean"]
    sc.pl.embedding(
        adata, basis="X_umap_tmp", color="cell_type",
        ax=ax, show=False,
        title=f"{fusion.upper()}\nsil={sil:.3f}, RNA \u03C1={rna_rho:.3f}",
    )

plt.tight_layout()
plt.show()

In [ ]:
# Summary table
summary = pd.DataFrame({
    fusion: {
        "Silhouette (cell type)": res["latent_metrics"].get("silhouette_labels", np.nan),
        "Silhouette (batch)": res["latent_metrics"].get("silhouette_batch", np.nan),
        "RNA Spearman (mean)": res["rna_recon"]["spearman_mean"],
        "RNA Spearman (var)": res["rna_recon"]["spearman_var"],
        "RNA RMSE": res["rna_recon"]["rmse"],
        "Protein Spearman (mean)": res["protein_recon"]["spearman_mean"],
        "Protein Spearman (var)": res["protein_recon"]["spearman_var"],
        "Protein RMSE": res["protein_recon"]["rmse"],
    }
    for fusion, res in fusion_results.items()
}).T

print("Fusion Strategy Comparison:")
print(summary.round(4).to_string())

In [ ]:
# Training curves comparison
fig, ax = plt.subplots(figsize=(7, 4))
for fusion, res in fusion_results.items():
    loss = res["history"]
    ax.plot(loss.index, loss["train_loss"], label=fusion.upper())
ax.set_xlabel("Epoch")
ax.set_ylabel("ELBO Loss")
ax.set_title("Training Convergence by Fusion Strategy")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Save and Load Model

cellPMVI models can be saved and loaded for later use. The saved model includes the neural network weights, training hyperparameters, and optionally the AnnData object.

In [ ]:
import os
os.makedirs("results", exist_ok=True)

model.save("results/cellpmvi_pbmc/", save_anndata=True)
print("Model saved.")

# Reload and verify
loaded_model = CellPMVI.load("results/cellpmvi_pbmc/")
latent_loaded = loaded_model.get_latent_representation()
print(f"Latent representations match: {np.allclose(latent, latent_loaded, atol=1e-5)}")

## Summary

In this tutorial we demonstrated the full cellPMVI workflow:

1. **Data preparation** — Loaded a CITE-seq PBMC dataset with RNA and 14 surface proteins, performed HVG selection and cell type annotation using protein markers
2. **Model training** — Trained cellPMVI with Product-of-Experts fusion to learn a joint RNA-protein latent space
3. **Latent space** — Visualized the integrated latent space showing cell type separation and batch mixing
4. **Reconstruction** — Assessed posterior predictive quality via Spearman correlation for both RNA and protein
5. **Generative sampling** — Generated synthetic cells from the prior distribution
6. **Fusion comparison** — Compared PoE, MoE, and independent (None) fusion strategies

### Key cellPMVI features

- **Multi-modal integration** — Separate encoders/decoders per modality with shared latent space
- **Flexible fusion** — PoE (sharper posteriors) vs MoE (robust to missing modalities) vs None (cross-modal reconstruction)
- **Latent distributions** — Normal or Laplace priors (`latent_distribution="lp"`)
- **scvi-tools compatible** — Built on `BaseModelClass` with standard `setup_anndata` / `train` / `save` / `load` workflow